In [0]:
%run ../../common/_utils_config

In [0]:
%run ../../common/_utils_reference_data

In [0]:
%run ../../bronze/utils/_utils_bronze_framework

In [0]:
%run ../../silver/utils/_utils_silver_contracts

In [0]:
%run ../utils/_utils_monitoring

In [0]:
failures = []


def check(description, condition):
    print(("✅ " if condition else "❌ ") + description)
    if not condition:
        failures.append(description)


views = refresh_monitoring()

for name in ["source_catalog"] + views:
    try:
        rows = spark.table(monitoring_object(name)).count()
        check(f"{name} can be queried ({rows} rows)", True)
    except Exception as e:
        check(f"{name} can be queried: {str(e)[:200]}", False)

catalog_sources = {r["source_name"] for r in spark.table(monitoring_object("source_catalog")).collect()}
check("source_catalog has every source spec", catalog_sources == set(list_source_specs()))

health = spark.table(monitoring_object("source_health"))
display(health)
check("source_health has one row per source", health.count() == len(catalog_sources))
check("every source has a known health",
      {r["health"] for r in health.collect()} <= {"ok", "delayed", "errors", "blocked", "never_loaded"})

freshness = spark.table(monitoring_object("table_freshness"))
display(freshness)
check("table_freshness lists the silver tables",
      {"data_source", "location", "electricity_price", "weather_observation_hourly"}
      <= {r["table_name"] for r in freshness.collect()})

overview = spark.table(monitoring_object("system_overview"))
display(overview)
check("system_overview returns exactly one row", overview.count() == 1)

silver_mappings = {r["mapping"] for r in spark.table(monitoring_object("silver_daily")).select("mapping").distinct().collect()}
check("silver_daily contains the REE mapping", "ree__electricity_price" in silver_mappings)

In [0]:
# Readable health summary (which sources are blocked, delayed or failing, and why)
health_rows = print_source_health()
check("print_source_health returns every source", len(health_rows) == len(catalog_sources))

In [0]:
if failures:
    raise AssertionError(f"{len(failures)} check(s) failed: {failures}")
print("✅ All monitoring checks passed")